# Downtown Toronto Hospitals Data Collection

This notebook contains the code to scrape:
1. **Qualitative Data (News)**: Using Google News RSS feeds to find articles related to downtown Toronto hospitals from the last 5 years.
2. **Quantitative Data (Open Data)**: Using the `opendatatoronto` library to pull healthcare institutional data, and highlighting how to analyze metrics from the last 5 years.

In [49]:
import pip
def install(package):
    if hasattr(pip, 'main'):
        pip.main(['install', package])
    else:
        pip._internal.main(['install', package])

# Install necessary packages if they don't exist
try:
    import feedparser
    import requests
except ImportError: 
    install('feedparser')
    install('requests')
    import feedparser
    import requests


## 1. Qualitative Data: Scraping News (Last 5 Years)
We use the Google News RSS feed, which allows searching by query, and supports the `after:` and `before:` date operators.

In [50]:
import urllib.parse
import feedparser
import pandas as pd
from datetime import datetime, timedelta
import time

# List of major downtown Toronto hospitals
downtown_hospitals = [
    "Toronto General Hospital",
    "Mount Sinai Hospital Toronto",
    "St. Michael's Hospital Toronto",
    "SickKids Hospital Toronto",
    "Toronto Western Hospital",
    "Princess Margaret Cancer Centre",
    "Women's College Hospital"
]

def _format_df(df):
    """Standardize dataframe format: date first, sorted columns, no scraping indicator columns."""
    if df.empty:
        return pd.DataFrame(columns=["date"])

    date_candidates = [c for c in df.columns if "date" in c.lower()]
    if "date" not in df.columns and date_candidates:
        df = df.rename(columns={date_candidates[0]: "date"})

    if "date" not in df.columns:
        df["date"] = pd.NaT

    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    # Remove scraping-indicator fields if present
    drop_cols = [c for c in df.columns if c.lower() in {"keyword", "category"}]
    if drop_cols:
        df = df.drop(columns=drop_cols)

    other_cols = sorted([c for c in df.columns if c != "date"])
    return df[["date"] + other_cols].sort_values(by="date", ascending=False).reset_index(drop=True)

def scrape_google_news(query, start_date, end_date):
    """
    Scrapes Google News RSS for a given query and date range.
    Dates should be in YYYY-MM-DD format.
    """
    search_query = f'"{query}" after:{start_date} before:{end_date}'
    url_encoded_query = urllib.parse.quote(search_query)
    rss_url = f"https://news.google.com/rss/search?q={url_encoded_query}&hl=en-CA&gl=CA&ceid=CA:en"

    feed = feedparser.parse(rss_url)
    articles = []

    for entry in feed.entries:
        articles.append({
            "hospital": query,
            "title": entry.title,
            "source": entry.source.title if hasattr(entry, "source") else "Unknown",
            "link": entry.link,
            "published_date": entry.published,
        })

    return articles

# Define the last 5 years date range
end_date = datetime.now()
start_date = end_date - timedelta(days=5 * 365)

start_date_str = start_date.strftime("%Y-%m-%d")
end_date_str = end_date.strftime("%Y-%m-%d")

all_qualitative_data = []
for hospital in downtown_hospitals:
    all_qualitative_data.extend(scrape_google_news(hospital, start_date_str, end_date_str))
    time.sleep(1.5)

news_df = pd.DataFrame(all_qualitative_data)
news_df = _format_df(news_df)
news_df


,date,hospital,link,source,title
0,2026-03-13 15:11:56,Princess Margaret Cancer Centre,https://news.google.com/rss/articles/CBMiW0FVX...,Oncodaily,Nihar Desai: A Remarkable Year at Princess Mar...
1,2026-03-10 21:51:13,Toronto General Hospital,https://news.google.com/rss/articles/CBMia0FVX...,UHN Foundation,"For the first time, UHN’s Toronto General Hosp..."
2,2026-02-26 08:00:00,Toronto Western Hospital,https://news.google.com/rss/articles/CBMikAFBV...,Global News,Toronto’s UHN ranked 2nd best Hospital in the ...
3,2026-02-26 08:00:00,Toronto General Hospital,https://news.google.com/rss/articles/CBMikAFBV...,Global News,Toronto’s UHN ranked 2nd best Hospital in the ...
4,2026-02-25 08:00:00,Toronto General Hospital,https://news.google.com/rss/articles/CBMidEFVX...,UHN Foundation,Proud to be named #2 in the world - UHN Founda...
...,...,...,...,...,...
407,2021-04-16 07:00:00,Toronto Western Hospital,https://news.google.com/rss/articles/CBMi-gFBV...,canada.ca,Noventa Energy Partners using innovative techn...
408,2021-03-24 07:00:00,Toronto Western Hospital,https://news.google.com/rss/articles/CBMijAFBV...,CBC,Thousands of Toronto hospital staff haven't go...
409,2021-03-23 07:00:00,Toronto Western Hospital,https://news.google.com/rss/articles/CBMiZ0FVX...,UHN Foundation,UHN Foundation is here - UHN Foundation
410,2021-03-18 07:00:00,Toronto General Hospital,https://news.google.com/rss/articles/CBMi3AFBV...,Toronto Star,Outbreak declared in Toronto General Hospital ...


## 2. Quantitative Data: Toronto Open Data & Other Sources
We can query the open data portal for Toronto to find datasets related to healthcare institutions and outbreaks (a common quantifiable metric for hospitals).

In [51]:
import pandas as pd
import requests


def _format_df(df):
    """Standardize dataframe format: date first, sorted columns, no scraping indicator columns."""
    if df.empty:
        return pd.DataFrame(columns=["date"])

    date_candidates = [c for c in df.columns if "date" in c.lower()]
    if "date" not in df.columns and date_candidates:
        df = df.rename(columns={date_candidates[0]: "date"})

    if "date" not in df.columns:
        df["date"] = pd.NaT

    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    drop_cols = [c for c in df.columns if c.lower() in {"keyword", "category", "_id"}]
    if drop_cols:
        df = df.drop(columns=drop_cols)

    other_cols = sorted([c for c in df.columns if c != "date"])
    return df[["date"] + other_cols].sort_values(by="date", ascending=False).reset_index(drop=True)


base_url = "https://ckan0.cf.opendata.inter.prod-toronto.ca"
search_url = f"{base_url}/api/3/action/package_search"
params = {"q": "healthcare"}
package_search_res = requests.get(search_url, params=params).json()

healthcare_packages = pd.DataFrame(package_search_res["result"]["results"])

outbreak_pkg_id = "outbreaks-in-toronto-healthcare-institutions"
outbreaks_df = pd.DataFrame()

try:
    package_url = f"{base_url}/api/3/action/package_show"
    package = requests.get(package_url, params={"id": outbreak_pkg_id}).json()

    datastore_resources = [r for r in package["result"]["resources"] if r.get("datastore_active")]

    if datastore_resources:
        data_resource = datastore_resources[0]
        data_url = f"{base_url}/api/3/action/datastore_search"
        data_params = {"id": data_resource["id"], "limit": 100000}
        data_res = requests.get(data_url, params=data_params).json()
        raw_df = pd.DataFrame(data_res["result"]["records"])

        date_cols = [c for c in raw_df.columns if "date" in c.lower()]
        if date_cols:
            main_date_col = date_cols[0]
            raw_df[main_date_col] = pd.to_datetime(raw_df[main_date_col], errors="coerce")
            raw_df = raw_df.rename(columns={main_date_col: "date"})

            try:
                five_years_ago_ts = pd.Timestamp(start_date)
                if raw_df["date"].dt.tz is not None:
                    five_years_ago_ts = five_years_ago_ts.tz_localize(raw_df["date"].dt.tz)
                raw_df = raw_df[raw_df["date"] >= five_years_ago_ts]
            except NameError:
                pass
        else:
            raw_df["date"] = pd.NaT

        outbreaks_df = _format_df(raw_df)
        # Keep only outbreaks at downtown Toronto hospitals
        if "Institution Name" in outbreaks_df.columns:
            outbreaks_df = outbreaks_df[outbreaks_df["Institution Name"].isin(downtown_hospitals)].reset_index(drop=True)
except Exception:
    outbreaks_df = pd.DataFrame(columns=["date"])

outbreaks_df


,date,Active,Causative Agent-1,Causative Agent-2,Date Declared Over,Institution Address,Institution Name,Outbreak Setting,Type of Outbreak


## 3. Web Scraping Alternative (Optional)
If APIs are insufficient, we can scrape qualitative text from hospital pages (news, annual reports, quality reports) and extract key signals related to hospital quality. This version captures snippets tied to funding, wait times, staffing, patient safety, and access/equity.

In [52]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime, timezone

QUALITY_KEYWORDS = {
    "funding": [
        "funding", "investment", "budget", "capital", "grant", "donation", "foundation"
    ],
    "wait_time": [
        "wait time", "waiting time", "er wait", "emergency wait", "surgical backlog", "queue"
    ],
    "staffing": [
        "staffing", "nurse", "physician", "hiring", "vacancy", "retention", "workforce"
    ],
    "quality_safety": [
        "patient safety", "infection", "readmission", "mortality", "quality improvement", "accreditation"
    ],
    "access_equity": [
        "equity", "access", "underserved", "indigenous", "language access", "mental health access"
    ],
}

def _format_df(df):
    """Standardize dataframe format: date first, sorted columns, no scraping indicator columns."""
    if df.empty:
        return pd.DataFrame(columns=["date"])

    date_candidates = [c for c in df.columns if "date" in c.lower()]
    if "date" not in df.columns and date_candidates:
        df = df.rename(columns={date_candidates[0]: "date"})

    if "date" not in df.columns:
        df["date"] = pd.NaT

    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    drop_cols = [c for c in df.columns if c.lower() in {"keyword", "category"}]
    if drop_cols:
        df = df.drop(columns=drop_cols)

    other_cols = sorted([c for c in df.columns if c != "date"])
    return df[["date"] + other_cols].sort_values(by="date", ascending=False).reset_index(drop=True)

def _clean_text(text):
    return re.sub(r"\s+", " ", text).strip()

def scrape_hospital_quality_signals(url, keyword_map=QUALITY_KEYWORDS):
    """
    Scrape a hospital webpage and return snippet-level qualitative rows.
    Output intentionally excludes scraping metadata columns (keyword/category).
    """
    try:
        headers = {"User-Agent": "Mozilla/5.0"}
        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")
        text_nodes = soup.find_all(["h1", "h2", "h3", "h4", "p", "li"])

        snippets = []
        seen = set()
        for node in text_nodes:
            txt = _clean_text(node.get_text(" ", strip=True))
            if len(txt) < 40 or txt in seen:
                continue
            seen.add(txt)
            snippets.append(txt)

        collected_date = pd.Timestamp(datetime.now(timezone.utc).date())
        rows = []
        for snippet in snippets:
            lower_snippet = snippet.lower()
            matched = any(
                word in lower_snippet
                for words in keyword_map.values()
                for word in words
            )
            if matched:
                rows.append({
                    "date": collected_date,
                    "source_url": url,
                    "snippet": snippet,
                })

        return pd.DataFrame(rows, columns=["date", "source_url", "snippet"])
    except Exception as e:
        print(f"Failed to scrape {url}: {e}")
        return pd.DataFrame(columns=["date", "source_url", "snippet"])

def scrape_multiple_hospital_pages(urls):
    frames = [scrape_hospital_quality_signals(url) for url in urls]
    frames = [f for f in frames if not f.empty]
    if not frames:
        return pd.DataFrame(columns=["date", "source_url", "snippet"])

    qualitative_df = pd.concat(frames, ignore_index=True).drop_duplicates().reset_index(drop=True)
    return _format_df(qualitative_df)

hospital_pages = [
    "https://www.uhn.ca/corporate/News",
    "https://sunnybrook.ca/content/?page=about-us-newsroom",
]

qualitative_df = scrape_multiple_hospital_pages(hospital_pages)

qualitative_df


Failed to scrape https://sunnybrook.ca/content/?page=about-us-newsroom: 404 Client Error: Not Found for url: https://sunnybrook.ca/content/?page=about-us-newsroom


,date,snippet,source_url
0,2026-03-15,Patients & Visitors Patients & Visitors At UHN...,https://www.uhn.ca/corporate/News
1,2026-03-15,Programs & Services Programs & Services Our UH...,https://www.uhn.ca/corporate/News
2,2026-03-15,Programs & Services Our UHN programs and servi...,https://www.uhn.ca/corporate/News
3,2026-03-15,Our UHN programs and services are among the mo...,https://www.uhn.ca/corporate/News
4,2026-03-15,About UHN About UHN University Health Network ...,https://www.uhn.ca/corporate/News


In [53]:
# Delete all rows with date 2026-03-14 or 2026-03-15
from datetime import date

exclude_dates = [date(2026, 3, 14), date(2026, 3, 15)]

def drop_dates(df, date_col="date"):
    if df is None or df.empty or date_col not in df.columns:
        return df
    ser = pd.to_datetime(df[date_col], errors="coerce")
    mask = ser.dt.date.isin(exclude_dates)
    return df.loc[~mask].reset_index(drop=True)

news_df = drop_dates(news_df)
outbreaks_df = drop_dates(outbreaks_df)
qualitative_df = drop_dates(qualitative_df)

print(f"Excluded rows with dates {exclude_dates}")

Excluded rows with dates [datetime.date(2026, 3, 14), datetime.date(2026, 3, 15)]


In [54]:
# 4. Compile all dataframes and sort by date
import pandas as pd


def _prepare_for_merge(df, name):
    print(f"Processing {name}...")

    if df is None or df.empty:
        print(f"- {name}: 0 articles")
        return pd.DataFrame(columns=["date", "dataset"])

    merged_df = df.copy()

    if "date" not in merged_df.columns:
        date_candidates = [c for c in merged_df.columns if "date" in c.lower()]
        if date_candidates:
            merged_df = merged_df.rename(columns={date_candidates[0]: "date"})
        else:
            merged_df["date"] = pd.NaT

    merged_df["date"] = pd.to_datetime(merged_df["date"], errors="coerce")

    # Remove scraping-indicator columns if present
    drop_cols = [c for c in merged_df.columns if c.lower() in {"keyword", "category"}]
    if drop_cols:
        merged_df = merged_df.drop(columns=drop_cols)

    merged_df["dataset"] = name
    print(f"- {name}: {len(merged_df)} articles")
    return merged_df

frames = [
    _prepare_for_merge(news_df, "news_df"),
    _prepare_for_merge(outbreaks_df, "outbreaks_df"),
    _prepare_for_merge(qualitative_df, "qualitative_df"),
]

compiled_df = pd.concat(frames, ignore_index=True, sort=False)
compiled_df = compiled_df.sort_values(by="date", ascending=False).reset_index(drop=True)

ordered_cols = ["date"] + sorted([c for c in compiled_df.columns if c != "date"])
compiled_df = compiled_df[ordered_cols]

# Drop dataset and link columns
compiled_df = compiled_df.drop(columns=[c for c in ["dataset", "link"] if c in compiled_df.columns])

print(f"\nTotal compiled records: {len(compiled_df)}")
print(compiled_df)
compiled_df

Processing news_df...
- news_df: 412 articles
Processing outbreaks_df...
- outbreaks_df: 0 articles
Processing qualitative_df...
- qualitative_df: 0 articles

Total compiled records: 412
                   date  dataset                         hospital  \
0   2026-03-13 15:11:56  news_df  Princess Margaret Cancer Centre   
1   2026-03-10 21:51:13  news_df         Toronto General Hospital   
2   2026-02-26 08:00:00  news_df         Toronto Western Hospital   
3   2026-02-26 08:00:00  news_df         Toronto General Hospital   
4   2026-02-25 08:00:00  news_df         Toronto Western Hospital   
..                  ...      ...                              ...   
407 2021-04-16 07:00:00  news_df         Toronto Western Hospital   
408 2021-03-24 07:00:00  news_df         Toronto Western Hospital   
409 2021-03-23 07:00:00  news_df         Toronto Western Hospital   
410 2021-03-18 07:00:00  news_df         Toronto General Hospital   
411 2021-03-18 07:00:00  news_df         Toronto Gener

C:\Users\Sihao\AppData\Local\Temp\ipykernel_3748\2486316702.py:38: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  compiled_df = pd.concat(frames, ignore_index=True, sort=False)


,date,dataset,hospital,link,source,title
0,2026-03-13 15:11:56,news_df,Princess Margaret Cancer Centre,https://news.google.com/rss/articles/CBMiW0FVX...,Oncodaily,Nihar Desai: A Remarkable Year at Princess Mar...
1,2026-03-10 21:51:13,news_df,Toronto General Hospital,https://news.google.com/rss/articles/CBMia0FVX...,UHN Foundation,"For the first time, UHN’s Toronto General Hosp..."
2,2026-02-26 08:00:00,news_df,Toronto Western Hospital,https://news.google.com/rss/articles/CBMikAFBV...,Global News,Toronto’s UHN ranked 2nd best Hospital in the ...
3,2026-02-26 08:00:00,news_df,Toronto General Hospital,https://news.google.com/rss/articles/CBMikAFBV...,Global News,Toronto’s UHN ranked 2nd best Hospital in the ...
4,2026-02-25 08:00:00,news_df,Toronto Western Hospital,https://news.google.com/rss/articles/CBMiswFBV...,CityNews Toronto,Newsweek ranks UHN’s Toronto General the secon...
...,...,...,...,...,...,...
407,2021-04-16 07:00:00,news_df,Toronto Western Hospital,https://news.google.com/rss/articles/CBMi-gFBV...,canada.ca,Noventa Energy Partners using innovative techn...
408,2021-03-24 07:00:00,news_df,Toronto Western Hospital,https://news.google.com/rss/articles/CBMijAFBV...,CBC,Thousands of Toronto hospital staff haven't go...
409,2021-03-23 07:00:00,news_df,Toronto Western Hospital,https://news.google.com/rss/articles/CBMiZ0FVX...,UHN Foundation,UHN Foundation is here - UHN Foundation
410,2021-03-18 07:00:00,news_df,Toronto General Hospital,https://news.google.com/rss/articles/CBMi3AFBV...,Toronto Star,Outbreak declared in Toronto General Hospital ...


In [55]:
# 5. Standardize all dataframes into: date, title, body, source
from urllib.parse import urlparse

def _build_outbreak_body(row):
    parts = [f"Outbreak at {row['Institution Name']} ({row['Institution Address']})."]
    parts.append(f"Setting: {row['Outbreak Setting']}.")
    parts.append(f"Type: {row['Type of Outbreak']}.")
    parts.append(f"Active: {row['Active']}.")
    if pd.notna(row.get("Causative Agent-2")) and row["Causative Agent-2"] not in (None, "None"):
        parts.append(f"Causative Agent 2: {row['Causative Agent-2']}.")
    if pd.notna(row.get("Date Declared Over")) and row["Date Declared Over"] not in (None, "None"):
        parts.append(f"Date Declared Over: {row['Date Declared Over']}.")
    return " ".join(parts)

def _domain_to_name(url):
    try:
        host = urlparse(url).hostname or ""
        domain = host.replace("www.", "")
        return domain.split(".")[0].upper()
    except Exception:
        return "Unknown"

# --- Outbreaks ---
ob = outbreaks_df[outbreaks_df["Causative Agent-1"].notna()].copy()
std_outbreaks = pd.DataFrame({
    "date": ob["date"],
    "title": ob["Causative Agent-1"],
    "body": ob.apply(_build_outbreak_body, axis=1),
    "source": "Toronto Public Health",
})

# --- News ---
std_news = pd.DataFrame({
    "date": news_df["date"],
    "title": news_df["title"],
    "body": news_df["title"],
    "source": news_df["source"],
})

# --- Qualitative web scrapes ---
std_qual = pd.DataFrame({
    "date": qualitative_df["date"],
    "title": qualitative_df["source_url"].apply(_domain_to_name),
    "body": qualitative_df["snippet"],
    "source": qualitative_df["source_url"].apply(_domain_to_name),
})

# --- Combine ---
standardized_df = (
    pd.concat([std_outbreaks, std_news, std_qual], ignore_index=True)
    .sort_values("date", ascending=False)
    .reset_index(drop=True)
)

print(f"Standardized records: {len(standardized_df)}")
standardized_df

Standardized records: 412


,date,title,body,source
0,2026-03-13 15:11:56,Nihar Desai: A Remarkable Year at Princess Mar...,Nihar Desai: A Remarkable Year at Princess Mar...,Oncodaily
1,2026-03-10 21:51:13,"For the first time, UHN’s Toronto General Hosp...","For the first time, UHN’s Toronto General Hosp...",UHN Foundation
2,2026-02-26 08:00:00,Toronto’s UHN ranked 2nd best Hospital in the ...,Toronto’s UHN ranked 2nd best Hospital in the ...,Global News
3,2026-02-26 08:00:00,Toronto’s UHN ranked 2nd best Hospital in the ...,Toronto’s UHN ranked 2nd best Hospital in the ...,Global News
4,2026-02-25 08:00:00,Newsweek ranks UHN’s Toronto General the secon...,Newsweek ranks UHN’s Toronto General the secon...,CityNews Toronto
...,...,...,...,...
407,2021-04-16 07:00:00,Noventa Energy Partners using innovative techn...,Noventa Energy Partners using innovative techn...,canada.ca
408,2021-03-24 07:00:00,Thousands of Toronto hospital staff haven't go...,Thousands of Toronto hospital staff haven't go...,CBC
409,2021-03-23 07:00:00,UHN Foundation is here - UHN Foundation,UHN Foundation is here - UHN Foundation,UHN Foundation
410,2021-03-18 07:00:00,Outbreak declared in Toronto General Hospital ...,Outbreak declared in Toronto General Hospital ...,Toronto Star
